# Dataset Preparation + Preprocessing

### import libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

## Loading the Curated COVID-19 Posterior-Anterior Chest X-Ray Dataset.

In [ ]:
# Constants
IMAGE_SIZE = 224
DATA_DIR = 'Curated_X-Ray_Dataset/'
CLASSES = ['COVID', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']
IMG_PER_CLASS_LIMIT = None

In [ ]:
# Data containers
X = []
y = []

In [ ]:
# Load images
for label in CLASSES:
    path = os.path.join(DATA_DIR, label)
    images = os.listdir(path)

    for i, img_name in enumerate(tqdm(images, desc=f"Loading {label}")):
        if IMG_PER_CLASS_LIMIT and i >= IMG_PER_CLASS_LIMIT:
            break
        try:
            img = cv2.imread(os.path.join(path, img_name))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
            X.append(img)
            y.append(label)
        except Exception as e:
            print(f"Failed on image: {img_name}, reason: {e}")

Loading Pneumonia-Viral: 100%|█████████████████████████████████████████████████████| 1656/1656 [00:19<00:00, 85.62it/s]


In [ ]:
# Convert to NumPy
X = np.array(X)
y = np.array(y)

In [ ]:
# Normalize
X = X / 255.0
# We divide by 255 to scale everything between 0 and 1.

In [ ]:
# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y) # Converts labels like "COVID", "NORMAL" → numbers like 0, 1, 2, 3.
y_categorical = to_categorical(y_encoded)

In [ ]:
# Save label mapping for later use
label_map = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label Mapping:", label_map)

Label Mapping: {'COVID': 0, 'Normal': 1, 'Pneumonia-Bacterial': 2, 'Pneumonia-Viral': 3}


### Split the Data

In [ ]:
# Split into train (70%), val (15%), test (15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y_categorical, test_size=0.3, stratify=y_encoded, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=np.argmax(y_temp, axis=1), random_state=42)

# Show shapes
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (6446, 224, 224, 3)
Val: (1381, 224, 224, 3)
Test: (1382, 224, 224, 3)


In [ ]:
np.save("X_train.npy", X_train)
np.save("X_val.npy", X_val)
np.save("X_test.npy", X_test)
np.save("y_train.npy", y_train)
np.save("y_val.npy", y_val)
np.save("y_test.npy", y_test)
